# Prétraitement de la base E3 : construction de la base nettoyée

Ce notebook définit une fonction de prétraitement `load_and_clean(path)` pour 
l'échantillon 3 (proches).

Objectifs :
- charger la base brute `Sample3_concat.csv` ;
- supprimer les colonnes techniques du questionnaire (date, langue, etc.) ;
- regrouper et nettoyer plusieurs variables :
  - activité professionnelle du répondant et du proche,
  - psychothérapies du proche,
  - diagnostics psychiatriques du proche ;
- créer une variable `Trouble_Catégorisé` composée de grandes catégories 
  diagnostiques (Depression, Anxiety, Bipolar disorder, etc.).

La base nettoyée produite par `load_and_clean` est ensuite réutilisée dans les 
notebooks :

- `BERTopic E3.ipynb` (modélisation de topics),
- éventuelles analyses descriptives spécifiques à l'échantillon 3.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata


def load_and_clean(path):

        """
        Charge et nettoie la base de l'échantillon 3 (proches).

        Étapes principales :
        - lecture du fichier CSV brut ;
        - suppression des colonnes techniques (date de soumission, page, langue,
        consentement, tête de série) ;
        - fusion des colonnes d'activité professionnelle du répondant et du proche
        en deux variables lisibles :
            * `Activité professionnelle (répondant)`
            * `Activité professionnelle (proche)` ;
        - regroupement des réponses aux psychothérapies du proche dans une variable
        textuelle (`Psychothérapies du proche`) puis standardisation dans 
        `Psychothérapie standardisée` ;
        - classification des diagnostics psychiatriques du proche en grandes
        catégories (Depression, Anxiety, Bipolar disorder, Addiction disorder,
        Eating disorder, Psychotic disorder, Cognitive disorder, etc.) ;
        - création de la variable `Trouble_Catégorisé` à partir du texte libre.

        Paramètres
        ----------
        path : str
            Chemin du fichier CSV brut (ex. "Data/Sample3_concat.csv").

        Retour
        ------
        df : pandas.DataFrame
            Base nettoyée et enrichie, prête pour l'analyse thématique et les
            analyses descriptives.
        """

    df = pd.read_csv(path, sep = ';')

    # --- Suppression des colonnes techniques du questionnaire ---
    df.drop(columns=['Date de soumission'], inplace=True)
    df.drop(columns=['Dernière page'], inplace=True)
    df.drop(columns=['Langue de départ'], inplace=True)
    df.drop(columns=["J'accepte"], inplace=True)
    df.drop(columns=["Tête de série"], inplace=True)

    # --- Fusion des informations d'activité professionnelle (répondant / proche) ---
    # Colonnes du répondant
    col_repondant_std = "Quelle est votre activité professionnel...e ou la dernière activité professionnelle que vous ayez exercé?"
    col_repondant_autre = "Quelle est votre activité professionn...dernière activité professionnelle que vous ayez exercé? [Autre]"
    # Colonnes du proche
    col_proche_std = "Quelle est l'activité professionnelle ou l... dernière activité professionnelle que votre proche ait exercé?"
    col_proche_autre = "Quelle est l'activité professionnelle ou...e activité professionnelle que votre proche ait exercé? [Autre]"
    # Fonctions fusion_repondant / fusion_proche
    # Création des colonnes "Activité professionnelle (répondant)" et "(proche)"
    # + drop des colonnes d'origine
    
    # Fusion des informations répondant
    def fusion_repondant(row):
        std = str(row[col_repondant_std]).strip() if pd.notna(row[col_repondant_std]) else ""
        autre = str(row[col_repondant_autre]).strip() if pd.notna(row[col_repondant_autre]) else ""
        if std and autre:
            return f"{std}, {autre}"
        return std or autre or "Non renseigné"
    
    # Fusion des informations proche
    def fusion_proche(row):
        std = str(row[col_proche_std]).strip() if pd.notna(row[col_proche_std]) else ""
        autre = str(row[col_proche_autre]).strip() if pd.notna(row[col_proche_autre]) else ""
        if std and autre:
            return f"{std}, {autre}"
        return std or autre or "Non renseigné"
    
    # Création des colonnes fusionnées
    df["Activité professionnelle (répondant)"] = df.apply(fusion_repondant, axis=1)
    df["Activité professionnelle (proche)"] = df.apply(fusion_proche, axis=1)
    
    # Suppression des colonnes d'origine
    df.drop(columns=[
        col_repondant_std, col_repondant_autre,
        col_proche_std, col_proche_autre
    ], inplace=True)



# --- Construction de la variable "Psychothérapies du proche" ---
    # colonnes_therapies, col_suivi, col_autre
    # libelles_therapies
    # fonction fusion_therapies -> "Psychothérapies du proche"
    # standardiser_therapie -> "Psychothérapie standardisée"
    # categories_tableau + assigner_categorie (stat interne, pas nécessairement utilisée ailleurs)
    # + drop des colonnes psychothérapies d'origine

    
    # Création d'une colonne rassemblant les différentes psychothérapies
    # Sélection automatique des colonnes liées à la question
    colonnes_therapies = [col for col in df.columns if "psychothérapie" in col.lower() and "[autre" not in col.lower()]
    col_suivi = [col for col in df.columns if "Si votre proche a déjà suivi une psychothérapie" in col]
    col_autre = [col for col in df.columns if "psychothérapie" in col.lower() and "[Autre" in col]
    
    # Dictionnaire de mappage : colonne → nom simple à afficher
    libelles_therapies = {
        col: col.split("une")[-1].strip(" ]).") for col in colonnes_therapies if "jamais" not in col.lower()
    }
    
    
    # Création de la colonne fusionnée
    def fusion_therapies(row):
        if any("jamais" in col.lower() and str(row[col]).strip().lower() == "oui" for col in colonnes_therapies):
            return "Aucune psychothérapie"
        
        liste_therapies = [
            libelles_therapies[col]
            for col in libelles_therapies
            if str(row[col]).strip().lower() == "oui"
        ]
        
        return ", ".join(liste_therapies) if liste_therapies else "Non renseigné"
    
    df["Psychothérapies du proche"] = df.apply(fusion_therapies, axis=1)
    
    # Gestion de la colonne [Autre]
    if col_autre:
        df.rename(columns={col_autre[0]: "Psychothérapie - Autre"}, inplace=True)
    
    # Suppression des colonnes d'origine sauf [Autre]
    df.drop(columns=[col for col in colonnes_therapies if col not in col_autre and col not in col_suivi], inplace=True)
    
    
    def standardiser_therapie(texte):
        if not isinstance(texte, str) or not texte.strip():
            return "Non renseigné"
    
        texte = texte.lower()
    
        # Cas directs
        if "aucune" in texte or "jamais" in texte:
            return "Aucune psychothérapie"
        if "psychanalyse" in texte:
            return "Psychanalyse"
        if "tcc" in texte or "cognitive et comportementale" in texte:
            return "TCC"
        if "systemique" in texte or "familiale" in texte:
            return "Thérapie systémique"
        if "humaniste" in texte:
            return "Thérapie humaniste"
        if "soutien" in texte:
            return "Thérapie de soutien"
        if "emdr" in texte:
            return "EMDR"
        if "icv" in texte or "lifespan" in texte:
            return "Intégration du cycle de vie (ICV)"
        if "je ne connais pas" in texte or "don’t know" in texte:
            return "Psychothérapie inconnue"
        if "autre" in texte:
            return "Autre psychothérapie"
    
        # Réponses narratives/floues
        if "qu’est ce qui a été travaillé" in texte or len(texte.split()) > 10:
            return "Psychothérapie non précisée"
    
        return "Autre psychothérapie"
    
    # Application
    df["Psychothérapie standardisée"] = df["Psychothérapies du proche"].apply(standardiser_therapie)
    
    
    # Création d’un tableau normalisé avec les catégories cibles
    categories_tableau = {
        "No psychotherapy": ["aucune psychothérapie", "non renseigné"],
        "Psychoanalysis": ["psychanalyse"],
        "Cognitive behavior therapy": ["tcc", "thérapie cognitive et comportementale"],
        "Systemic therapy": ["thérapie systémique", "familiale"],
        "Humanist therapy": ["thérapie humaniste"],
        "Supportive therapy": ["thérapie de soutien"],
        "Eye Movement Desensitization and Reprocessing (EMDR)": ["emdr"],
        "Lifespan integration": ["icv", "lifespan"],
        "I don’t know the name of my therapy": ["je ne connais pas", "je ne sais pas", "don’t know"],
        "Other": ["autre", "psychothérapie non précisée"]
    }
    
    compteur = {cat: 0 for cat in categories_tableau}
    
    def assigner_categorie(texte):
        if not isinstance(texte, str):
            return
    
        texte = texte.lower()
    
        for categorie, mots_cles in categories_tableau.items():
            if any(mot in texte for mot in mots_cles):
                compteur[categorie] += 1
                return
    
        # Si aucun mot-clé détecté, classé comme "Other"
        compteur["Other"] += 1
    
    # Appliquer au texte brut ou standardisé
    df["Psychothérapie standardisée"].apply(assigner_categorie)
    df.drop(columns=["Psychothérapies du proche", "Psychothérapie - Autre"], inplace=True)



    # Création d'une colonne regroupant les troubles
    # Fonction pour supprimer les accents
    def remove_accents(text):
        return ''.join(
            c for c in unicodedata.normalize('NFD', text)
            if unicodedata.category(c) != 'Mn'
        )
    
    # Fonction de classification
    def classer_troubles(texte):
        if pd.isna(texte):
            return np.nan
    
        # Nettoyage du texte
        texte = texte.lower()
        texte = remove_accents(texte)
    
        categories = set()
    
        if any(mot in texte for mot in ["depression", "depressive", "deprime", "depressif", "ts", "edc"]):
            categories.add("Depression")
    
        if any(mot in texte for mot in ["anxiete", "angoisse", "anxiete generalisee", "stress", "anxieux", "tag"]):
            categories.add("Anxiety")
    
        if any(mot in texte for mot in ["bipolaire","bipolarite", "bi polarité", "bi polaire", "bi-polarité", "pmd"]):
            categories.add("Bipolar disorder")
    
        if any(mot in texte for mot in ["alcool", "alcoolisme", "alcoolique", "addiction", "addictions", "dependance", "drogue", "toxicomanie",
                                        "tabac", "tabagique"
                                       ]):
            categories.add("Addiction disorder")
    
        if any(mot in texte for mot in ["borderline", "personnalite", "manipule", "manipulateur", "trouble de perso"]):
            categories.add("Personality disorder")
    
        if any(mot in texte for mot in ["boulimie", "anorexie", "alimentaire", "alimentation", "tca"]):
            categories.add("Eating disorder")
    
        if any(mot in texte for mot in ["alzheimer", "declin", "memoire", "demence", "cognitif", "desorientation", "confusion",
                                       "parkinson", "ecriture et lecture lentes"]):
            categories.add("Cognitive disorder")
    
        if any(mot in texte for mot in [
            "schizophrenie", "schizophrène", "hallucination", "hallucinations", 
            "delire", "delires", "paranoia", "paranoiaque","paranoïde", "psychose", 
            "psychotique", "trouble psychotique", "dissociation", "dissociatif"
        ]):
            categories.add("Psychotic disorder")
    
        # Cas spéciaux ou flous
        if any(mot in texte for mot in [
            "sspt", "ptsd", "trauma", "toc", "hyperactivite", "insecurite", "trouble", "instabilite", 
            "dyspraxie", "hypersensibilite", "trouble obsessionnel compulsif", "lassitude", "burn out",
            "burnout"
        ]):
            categories.add("Other psychiatric disorder")
    
        if not categories:
            return "Other psychiatric disorder"
        return ", ".join(sorted(categories))
    
    
    # Application sur une colonne de texte libre
    df["Trouble_Catégorisé"] = df["Merci de nous indiquer ici le ou les trouble(s) psychiatriques dont souffre votre proche"].apply(classer_troubles)

    return df


## 2. Application du prétraitement et vérifications

Nous appliquons maintenant la fonction `load_and_clean` à la base brute 
`Sample3_concat.csv` pour obtenir la base nettoyée utilisée dans les analyses 
de l'échantillon 3.

Nous affichons ensuite quelques informations de base sur le DataFrame.

In [ ]:
# Application de la fonction de prétraitement à la base brute
df = load_and_clean("Data/Sample3_concat.csv")

# Aperçu des premières lignes de la base nettoyée
df.head()

,ID de la réponse,Genre,Genre [Autre],Age,Quel est votre niveau d'étude ?,Genre de votre proche,Genre de votre proche [Autre],Age de votre proche,Quel est le niveau d'étude de votre proche?,Merci de nous indiquer ici le ou les trouble(s) psychiatriques dont souffre votre proche,...,Que voudriez-vous voir changer dans le quotidien de votre proche ?,"Quelles sont les situations et/ou les évènements (e.g. situations de violences) qui peuvent éventuellement faire souffrir vos proches, pouvez-vous nous les expliquer ?","Si votre proche a déjà suivi une psychothérapie, qu’est-ce que cela lui a apporté selon vous ?","Si votre proche a déjà suivi une psychothérapie, qu’est ce qui a été travaillé avec le thérapeute ?","Qu’est-ce qui peut, selon vous, constituer un frein à l’amélioration de l’état de votre proche ?",Commentaire libre,Activité professionnelle (répondant),Activité professionnelle (proche),Psychothérapie standardisée,Trouble_Catégorisé
0,1,femme,NaN,28.0,Bac+2,homme,NaN,25.0,Bac,Trouble bipolaire/schizophrénie,...,"Le voir plus indépendant, avec plus d'entrain ...",Ne supporte pas d'entendre mon père s'énerver ...,Ne veut pas suivre de thérapie malgré mes cons...,NaN,"Le fait d'être à la maison souvent, de ne pas ...","Sujet difficile, mais j'espère que ça vous aid...",Employé,Etudiant,Aucune psychothérapie,"Bipolar disorder, Other psychiatric disorder, ..."
1,2,homme,NaN,22.0,Bac,homme,NaN,22.0,Brevet des collèges,"Il n’a pas été diagnostiqué, mais il présente ...",...,NaN,NaN,NaN,NaN,NaN,NaN,Etudiant,Employé,Aucune psychothérapie,Depression
2,3,femme,NaN,34.0,Supérieur au Bac+5,homme,NaN,35.0,Brevet des collèges,Anxiété,...,J'aimerais qu'il prenne plus de temps de lui m...,Je ne suis pas certaine si je comprends la que...,Mon mari refuse de suivre de la psychothérapie...,Nous discutons souvent ensemble pendant des he...,La maladie de quelqu'un proche de lui ou s'il ...,NaN,Employé,Ouvrier,Autre psychothérapie,Anxiety
3,4,homme,NaN,24.0,Brevet des collèges,femme,NaN,24.0,Brevet des collèges,TDAH,...,"Moins d'auto-critique, meilleure routine, outi...","Reproches constants, échecs répétés (études/tr...","Meilleure estime de soi, stratégies pour l'org...","Gestion du temps, techniques anti-procrastinat...","Manque de diagnostic/clarification, stigmatisa...",NaN,Ouvrier,Employé,Psychothérapie inconnue,Other psychiatric disorder
4,5,homme,NaN,38.0,Bac,homme,NaN,45.0,Bac,Dépression,...,Le voir reprendre du plaisir a vivre,Les échecs essentiellement,"Il est entrain d'en suivre une, il faut approf...",Je ne sais pas,Son manque de combativité,NaN,Employé,Employé,Thérapie de soutien,Depression


In [ ]:
# Vérification de la taille et des colonnes de la base nettoyée
df.shape, df.columns

Nous vérifions le nombre de lignes / colonnes et la liste des variables
présentes dans la base nettoyée.

## 3. Sauvegarde de la base nettoyée

La base nettoyée `df` est sauvegardée au format CSV afin d'être réutilisée dans 
les notebooks d'analyse de l'échantillon 3, notamment `BERTopic E3.ipynb`.

In [ ]:
# Sauvegarde de la base nettoyée pour réutilisation ultérieure
df.to_csv("Data/Sample3_concat_clean.csv", sep=",", index=False)